In [ ]:
# HODGE–HAAR EXPLICIT INTERTWINER CONTRACTION ENGINE v0.4
# =======================================================
# Self-contained Google Colab / Python block.
#
# This is the first version that evaluates actual color-index contraction
# networks rather than only center balance or local singlet multiplicities.
#
# CORE CAPABILITY
# ---------------
# For every integrated link with k U factors and k Ubar factors (k <= 3),
# use the exact unitary Weingarten formula
#
#   ∫ dU ∏_r U_{i_r j_r} ∏_s Ubar_{i'_s j'_s}
#     = Σ_{σ,τ ∈ S_k}
#       Wg_N(σ^{-1}τ)
#       ∏_r δ_{i_r,i'_{σ(r)}} δ_{j_r,j'_{τ(r)}}.
#
# The code:
#   * materializes every permutation intertwiner branch;
#   * carries exact rational functions of symbolic N;
#   * joins Kronecker deltas with an exact DSU/partition state;
#   * sums N^(number of free color loops);
#   * merges algebraically identical contraction states.
#
# SU(3) determinant primitive
# ---------------------------
# Also implemented explicitly:
#
#   ∫_{SU(3)} dU U_{i1j1}U_{i2j2}U_{i3j3}
#     = (1/3!) ε_{i1i2i3} ε_{j1j2j3}
#
# with εε expanded into its six signed permutation-delta contractions.
#
# Mixed determinant-plus-delta cases such as (a,b)=(4,1) are DETECTED and
# QUARANTINED, not faked.  They are the explicit target for v0.5.
#
# ACCEPTANCE TESTS
# ----------------
#  1. Exact k=1,2,3 Weingarten Gram inversion.
#  2. Plaquette Haar norm = 1.
#  3. Plaquette-deformation lemma derived directly:
#         <outer 6-link loop | χ_p | plaquette> = 1/N.
#  4. All 75 exact depth-2 one-plaquette endpoint trace networks are contracted.
#     Their exact global amplitudes collapse to {1:72, 2:3}.
#  5. All 48 direct cube histories contract to exactly N^-4 from the full
#     six-trace color network.
#  6. Their temporal resolvents then reproduce
#         c_N^square = -160/[N(N^2-1)^3],
#         alpha_N    =  640/[N(N^2-1)^3].
#  7. Structured depth-3 SU(3) endpoint candidates are classified into:
#         fully explicit v0.4 networks
#         vs mixed determinant channels deferred to v0.5.
#
# No GPU is required for this correctness layer.  The v0.2/v0.3 GPU generator
# remains the throughput front-end; v0.4 is the exact symbolic contraction back-end.

import itertools
import math
import time
from collections import defaultdict, Counter
from fractions import Fraction

import numpy as np
import sympy as sp

# ======================================================================================
# CONFIG / GATES
# ======================================================================================

L = 3
SEED_FACE = 0
Nsym = sp.symbols("N", integer=True, positive=True)
N_RANK = 3
MAX_BALANCED_K = 3

gates=[]
def gate(name,ok,detail=""):
    ok=bool(ok)
    gates.append((name,ok,str(detail)))
    print(("[PASS] " if ok else "[FAIL] ")+name+(f" :: {detail}" if detail!="" else ""))

# ======================================================================================
# PART I — PERMUTATIONS / WEINGARTEN FUNCTIONS
# ======================================================================================

print("="*112)
print("PART I — EXACT WEINGARTEN INTERTWINER TABLES")
print("="*112)

def perm_inverse(p):
    out=[0]*len(p)
    for i,j in enumerate(p):
        out[j]=i
    return tuple(out)

def perm_compose(p,q):
    return tuple(p[q[i]] for i in range(len(p)))

def perm_cycles(p):
    seen=[False]*len(p)
    c=0
    for i in range(len(p)):
        if not seen[i]:
            c+=1
            j=i
            while not seen[j]:
                seen[j]=True
                j=p[j]
    return c

def perm_sign(p):
    inv=0
    for i in range(len(p)):
        for j in range(i+1,len(p)):
            inv += (p[i]>p[j])
    return -1 if inv%2 else 1

_WG={}
def weingarten_table(k):
    """
    Return (permutations, inverse Gram matrix).
    G_{σ,τ} = N^{#cycles(σ^{-1}τ)}.
    """
    if k in _WG:
        return _WG[k]
    ps=list(itertools.permutations(range(k)))
    G=sp.Matrix([
        [Nsym**perm_cycles(perm_compose(perm_inverse(p),q)) for q in ps]
        for p in ps
    ])
    Gi=sp.simplify(G.inv())
    _WG[k]=(ps,G,Gi)
    return _WG[k]

for k in (1,2,3):
    ps,G,Gi=weingarten_table(k)
    gate(f"k={k}: exact Weingarten inverse satisfies G W = I",
         sp.simplify(G*Gi-sp.eye(math.factorial(k)))==sp.zeros(math.factorial(k)))

ps2,G2,W2=weingarten_table(2)
gate("k=2 identity Wg = 1/(N^2-1)",
     sp.simplify(W2[0,0]-1/(Nsym**2-1))==0,
     W2[0,0])
gate("k=2 transposition Wg = -1/[N(N^2-1)]",
     sp.simplify(W2[0,1]+1/(Nsym*(Nsym**2-1)))==0,
     W2[0,1])

print("k=2 Weingarten matrix:")
sp.pprint(W2)

# SU(3) epsilon primitive:
# integral coefficient is 1/6 times ε_i ε_j; εε has 6 signed delta permutations.
eps_perms=list(itertools.permutations(range(3)))
eps_coeffs=[sp.Rational(perm_sign(p),math.factorial(3)) for p in eps_perms]
gate("SU(3) pure determinant primitive has six signed permutation branches",
     len(eps_coeffs)==6 and sum(abs(c) for c in eps_coeffs)==1,
     eps_coeffs)
# det U = (1/6) ε_i ε_j UUU, so ∫ det U = (1/36)*(Σ ε^2)^2 = 1.
gate("SU(3) epsilon primitive gives integral det(U)=1",
     sp.Rational(1,36)*6*6==1)

# ======================================================================================
# PART II — CUBIC CELL COMPLEX + ORDERED WILSON LOOPS
# ======================================================================================

print("\n"+"="*112)
print("PART II — CUBIC CELL COMPLEX AND TRACE FACTORS")
print("="*112)

def shift(v,d,step,L):
    w=list(v)
    w[d]=(w[d]+step)%L
    return tuple(w)

def build_cubic_complex(L):
    verts=[(x,y,z) for x in range(L) for y in range(L) for z in range(L)]

    links=[]
    lid={}
    for v in verts:
        for d in range(3):
            lid[(v,d)]=len(links)
            links.append((v,d))

    planes=[(0,1),(0,2),(1,2)]
    faces=[]
    fid={}
    for v in verts:
        for a,b in planes:
            fid[(v,a,b)]=len(faces)
            faces.append((v,a,b))

    E,P=len(links),len(faces)
    B2=np.zeros((E,P),dtype=np.int8)

    for f,(v,a,b) in enumerate(faces):
        va=shift(v,a,+1,L)
        vb=shift(v,b,+1,L)
        B2[lid[(v,a)],f]+=1
        B2[lid[(va,b)],f]+=1
        B2[lid[(vb,a)],f]-=1
        B2[lid[(v,b)],f]-=1

    B3=np.zeros((P,len(verts)),dtype=np.int8)
    for c,v in enumerate(verts):
        vx=shift(v,0,+1,L)
        vy=shift(v,1,+1,L)
        vz=shift(v,2,+1,L)
        B3[fid[(vx,1,2)],c]+=1
        B3[fid[(v,1,2)],c]-=1
        B3[fid[(vy,0,2)],c]-=1
        B3[fid[(v,0,2)],c]+=1
        B3[fid[(vz,0,1)],c]+=1
        B3[fid[(v,0,1)],c]-=1

    incidence=(B2!=0)
    ADJ=(incidence.T.astype(np.int16)@incidence.astype(np.int16))>0

    return verts,links,faces,lid,fid,B2,B3,ADJ

verts,links,faces,lid,fid,B2,B3,ADJ=build_cubic_complex(L)
E,P=B2.shape
C=B3.shape[1]

gate("B2 B3 = 0",
     np.max(np.abs(B2.astype(np.int16)@B3.astype(np.int16)))==0)
gate("cubic face linked-neighborhood = 13 including self",
     np.all(ADJ.sum(axis=1)==13),
     Counter(ADJ.sum(axis=1).tolist()))

def face_steps(face,sign=+1):
    """
    Ordered oriented boundary of a plaquette.
    A step is (physical_link_id, traversal_direction ±1).
    """
    v,a,b=faces[int(face)]
    va=shift(v,a,+1,L)
    vb=shift(v,b,+1,L)
    st=[
        (lid[(v,a)],+1),
        (lid[(va,b)],+1),
        (lid[(vb,a)],-1),
        (lid[(v,b)],-1),
    ]
    if int(sign)<0:
        st=[(l,-d) for l,d in reversed(st)]
    return st

def conjugate_steps(st):
    return [(l,-d) for l,d in reversed(st)]

def chain_to_loop_steps(chain):
    """
    Convert a simple oriented ±1 link 1-cycle into one ordered loop.
    """
    outgoing={}
    indeg=Counter()
    for l,q in enumerate(np.asarray(chain,dtype=int)):
        if q==0:
            continue
        if abs(q)!=1:
            raise ValueError("not a simple ±1 chain")
        v,d=links[l]
        w=shift(v,d,+1,L)
        if q==+1:
            src,tgt,ori=v,w,+1
        else:
            src,tgt,ori=w,v,-1
        if src in outgoing:
            raise ValueError("multiple outgoing edges")
        outgoing[src]=(tgt,l,ori)
        indeg[tgt]+=1

    if not outgoing:
        return []

    if any(v!=1 for v in indeg.values()):
        raise ValueError("not a simple directed cycle")

    start=next(iter(outgoing))
    cur=start
    seen=set()
    st=[]
    while True:
        if cur in seen:
            if cur!=start:
                raise ValueError("cycle closes at wrong vertex")
            break
        seen.add(cur)
        if cur not in outgoing:
            raise ValueError("open path")
        tgt,l,ori=outgoing[cur]
        st.append((l,ori))
        cur=tgt

    if len(st)!=np.count_nonzero(chain):
        raise ValueError("disconnected cycle")
    return st

# ======================================================================================
# PART III — EXACT TRACE-NETWORK CONTRACTOR
# ======================================================================================

print("\n"+"="*112)
print("PART III — EXPLICIT TRACE-NETWORK CONTRACTION ENGINE")
print("="*112)

def canonical_partition(parent):
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]
            i=parent[i]
        return i
    roots=[find(i) for i in range(len(parent))]
    relabel={}
    out=[]
    for r in roots:
        if r not in relabel:
            relabel[r]=len(relabel)
        out.append(relabel[r])
    return tuple(out)

def apply_unions(state,pairs):
    n=len(state)
    parent=list(range(n))

    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x

    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb:
            parent[rb]=ra

    first={}
    for i,label in enumerate(state):
        if label in first:
            union(first[label],i)
        else:
            first[label]=i

    for a,b in pairs:
        union(int(a),int(b))

    return canonical_partition(parent)

def build_trace_network(trace_factors):
    """
    trace_factors = list of ordered Wilson loops.
    Each loop is a list[(link_id, direction ±1)].

    For negative traversal:
      (U^\dagger)_{ab} = \bar U_{ba}.
    Therefore the Ubar index pair is stored as (b,a).
    """
    U=defaultdict(list)
    UB=defaultdict(list)
    nvars=0

    for st in trace_factors:
        m=len(st)
        xs=list(range(nvars,nvars+m))
        nvars+=m

        for j,(link,direction) in enumerate(st):
            a=xs[j]
            b=xs[(j+1)%m]
            if int(direction)>0:
                U[int(link)].append((a,b))
            else:
                UB[int(link)].append((b,a))

    return nvars,U,UB

def balanced_local_branches(Ulist,Blist,Nvalue=None):
    """
    Explicit Weingarten branches for k U and k Ubar.
    Returns [(coefficient, delta_pairs), ...].
    """
    k=len(Ulist)
    assert len(Blist)==k
    if k>MAX_BALANCED_K:
        raise NotImplementedError(f"balanced k={k} > {MAX_BALANCED_K}")

    ps,G,W=weingarten_table(k)
    out=[]
    for si,sigma in enumerate(ps):
        for ti,tau in enumerate(ps):
            coeff=W[si,ti]
            if Nvalue is not None:
                coeff=sp.simplify(coeff.subs(Nsym,Nvalue))
            pairs=[]
            for r in range(k):
                pairs.append((Ulist[r][0],Blist[sigma[r]][0]))
                pairs.append((Ulist[r][1],Blist[tau[r]][1]))
            out.append((coeff,pairs))
    return out

def pure_epsilon_branches(items,Nrank):
    """
    ∫ U^N = (1/N!) ε(rows) ε(cols)
    and εε = Σ_perm sign(perm) ∏ δ(row_r,col_perm(r)).
    """
    assert len(items)==Nrank
    out=[]
    for p in itertools.permutations(range(Nrank)):
        coeff=sp.Rational(perm_sign(p),math.factorial(Nrank))
        pairs=[(items[r][0],items[p[r]][1]) for r in range(Nrank)]
        out.append((coeff,pairs))
    return out

def classify_local_occupancy(a,b,Nrank):
    if a==b:
        if a<=MAX_BALANCED_K:
            return "balanced"
        return "balanced_high"
    if (a,b)==(Nrank,0) or (a,b)==(0,Nrank):
        return "pure_epsilon"
    if (a-b)%Nrank==0:
        return "mixed_determinant"
    return "forbidden"

def contract_trace_network(trace_factors,Nrank=3):
    """
    Exact trace-network contraction.

    Returns:
      expression, metadata

    Balanced-only networks are symbolic rational functions of N.
    Networks containing pure epsilon primitives are evaluated exactly at N=Nrank.
    Mixed determinant-plus-delta links are explicitly marked unsupported.
    """
    nvars,U,UB=build_trace_network(trace_factors)
    active=sorted(set(U)|set(UB))

    local_types={}
    has_epsilon=False
    for l in active:
        a=len(U[l]); b=len(UB[l])
        typ=classify_local_occupancy(a,b,Nrank)
        local_types[l]=(a,b,typ)
        if typ in ("forbidden","balanced_high","mixed_determinant"):
            return None,{
                "status":"unsupported" if typ!="forbidden" else "zero",
                "link":l,
                "occupancy":(a,b),
                "type":typ,
                "local_types":local_types,
            }
        if typ=="pure_epsilon":
            has_epsilon=True

    Ncolor=sp.Integer(Nrank) if has_epsilon else Nsym
    states={tuple(range(nvars)):sp.Integer(1)}
    max_states=1
    branch_product=1

    for l in active:
        a,b,typ=local_types[l]

        if typ=="balanced":
            branches=balanced_local_branches(U[l],UB[l],
                                             Nvalue=(Nrank if has_epsilon else None))
        elif typ=="pure_epsilon":
            items=U[l] if a else UB[l]
            branches=pure_epsilon_branches(items,Nrank)
        else:
            raise RuntimeError("unreachable type")

        branch_product*=len(branches)
        new=defaultdict(lambda:sp.Integer(0))

        for st,c in states.items():
            for bc,pairs in branches:
                st2=apply_unions(st,pairs)
                new[st2]+=c*bc

        states={
            st:sp.factor(c)
            for st,c in new.items()
            if sp.simplify(c)!=0
        }
        max_states=max(max_states,len(states))

    total=sp.Integer(0)
    for st,c in states.items():
        free_colors=len(set(st))
        total += c*Ncolor**free_colors

    total=sp.factor(sp.cancel(sp.simplify(total)))
    return total,{
        "status":"exact",
        "nvars":nvars,
        "active_links":len(active),
        "final_partition_states":len(states),
        "max_partition_states":max_states,
        "raw_local_branch_product":branch_product,
        "has_epsilon":has_epsilon,
        "local_types":local_types,
    }

# ======================================================================================
# PART IV — FIRST-PRINCIPLES HAAR REGRESSIONS
# ======================================================================================

print("\n"+"="*112)
print("PART IV — FIRST-PRINCIPLES HAAR REGRESSIONS")
print("="*112)

# 1. Plaquette norm.
plaquette_norm,meta=contract_trace_network([
    face_steps(SEED_FACE,+1),
    face_steps(SEED_FACE,-1),
])
print("plaquette norm =",plaquette_norm)
gate("single fundamental plaquette Haar norm = 1",
     sp.simplify(plaquette_norm-1)==0,
     plaquette_norm)

# 2. Find an adjacent plaquette whose properly oriented attachment gives
#    a simple six-link outer loop, then derive the deformation lemma.
deform=None
for q in range(P):
    if q==SEED_FACE:
        continue
    for s in (-1,+1):
        chain=B2[:,SEED_FACE].astype(int)+s*B2[:,q].astype(int)
        if np.count_nonzero(chain)!=6 or np.max(np.abs(chain))!=1:
            continue
        try:
            outer=chain_to_loop_steps(chain)
            deform=(q,s,outer)
            break
        except ValueError:
            pass
    if deform:
        break

assert deform is not None
qface,qsign,outer_loop=deform
deform_amp,deform_meta=contract_trace_network([
    face_steps(SEED_FACE,+1),
    face_steps(qface,qsign),
    conjugate_steps(outer_loop),  # bra outer loop
])

print(f"deformation test: seed={SEED_FACE}, attached={qface} sign={qsign}, perimeter={len(outer_loop)}")
print("explicit contraction =",deform_amp)
gate("plaquette-deformation lemma derived explicitly: amplitude = 1/N",
     sp.simplify(deform_amp-1/Nsym)==0,
     deform_amp)
gate("deformation network has a unique final contraction state",
     deform_meta["final_partition_states"]==1,
     deform_meta)

# ======================================================================================
# PART V — STRUCTURED DEPTH-2 CORPUS: CONTRACT ALL EXACT ENDPOINT NETWORKS
# ======================================================================================

print("\n"+"="*112)
print("PART V — ALL DEPTH-2 EXACT ONE-PLAQUETTE ENDPOINT NETWORKS")
print("="*112)

def generate_structured_words(depth):
    wf=np.empty((1,0),dtype=np.int16)
    ws=np.empty((1,0),dtype=np.int8)
    flux=np.asarray(B2[:,SEED_FACE][None,:],dtype=np.int16)

    for d in range(depth):
        M=wf.shape[0]
        mask=np.broadcast_to(ADJ[SEED_FACE],(M,P)).copy()
        for j in range(d):
            mask |= ADJ[wf[:,j]]

        rows,fs=np.nonzero(mask)
        K=len(rows)
        rr=np.repeat(rows,2)
        ff=np.repeat(fs.astype(np.int16),2)
        ss=np.tile(np.asarray([-1,+1],dtype=np.int8),K)

        nwf=np.empty((2*K,d+1),dtype=np.int16)
        nws=np.empty((2*K,d+1),dtype=np.int8)
        if d:
            nwf[:,:d]=wf[rr]
            nws[:,:d]=ws[rr]
        nwf[:,d]=ff
        nws[:,d]=ss

        flux=flux[rr]+ss[:,None]*B2[:,ff].T.astype(np.int16)
        wf,ws=nwf,nws

    return wf,ws,flux

wf2,ws2,flux2=generate_structured_words(2)
gate("depth-2 structured corpus has 1028 actual linked words",
     len(wf2)==1028,
     len(wf2))

exact_endpoint_map={}
for f in range(P):
    exact_endpoint_map[tuple(B2[:,f].astype(int))]=(f,+1)
    exact_endpoint_map[tuple((-B2[:,f]).astype(int))]=(f,-1)

exact2=[]
for i,q in enumerate(flux2):
    ep=exact_endpoint_map.get(tuple(int(x) for x in q))
    if ep is not None:
        exact2.append((i,ep,wf2[i],ws2[i]))

gate("depth-2 exact endpoint corpus has 75 histories",
     len(exact2)==75,
     len(exact2))

amp_hist=Counter()
state_hist=Counter()
unsupported2=[]
offdiag_bad=[]

seed_neighbors=set(np.flatnonzero(ADJ[SEED_FACE]))

t0=time.time()
for i,(ef,es),wf,ws in exact2:
    factors=[face_steps(SEED_FACE,+1)]
    factors += [face_steps(int(f),int(s)) for f,s in zip(wf,ws)]
    factors += [face_steps(int(ef),-int(es))]   # bra endpoint

    amp,meta=contract_trace_network(factors,Nrank=N_RANK)
    if amp is None:
        unsupported2.append((i,ef,es,wf,ws,meta))
        continue

    amp_hist[str(sp.factor(amp))]+=1
    state_hist[meta["max_partition_states"]]+=1

    if ef!=SEED_FACE and ef not in seed_neighbors:
        offdiag_bad.append((ef,es,wf,ws,amp))

print("depth-2 exact global amplitude histogram:",dict(amp_hist))
print("depth-2 maximum DSU-state histogram:",dict(state_hist))
print(f"contracted 75 networks in {time.time()-t0:.3f}s")

gate("all 75 depth-2 exact endpoint networks are explicitly contractible",
     len(unsupported2)==0,
     len(unsupported2))
gate("depth-2 full Haar contraction histogram = 72x1 + 3x2",
     amp_hist==Counter({"1":72,"2":3}),
     dict(amp_hist))
gate("every depth-2 offdiagonal exact endpoint is local/shared-edge",
     len(offdiag_bad)==0,
     len(offdiag_bad))

# This is the important Fierz lesson:
# local invariant multiplicities >1 do NOT mean the final trace network has
# the same multiplicity.  Explicit permutation branches cancel/recombine.
gate("explicit contraction performs nontrivial Fierz recombination",
     any(k>1 for k in state_hist),
     dict(state_hist))

# ======================================================================================
# PART VI — CUBE: FULL SIX-TRACE COLOR NETWORK
# ======================================================================================

print("\n"+"="*112)
print("PART VI — ALL 48 DIRECT CUBE NETWORKS")
print("="*112)

def expected_cube_words(seed):
    out=[]
    for c in range(C):
        coeff=B3[:,c].astype(int)
        if coeff[seed]==0:
            continue

        coeff*=int(coeff[seed])  # seed coefficient +1
        ids=np.flatnonzero(coeff)
        others=[int(f) for f in ids if f!=seed]

        seed_edges=set(np.flatnonzero(B2[:,seed]))
        opposite=[]
        side=[]
        for f in others:
            if seed_edges.isdisjoint(set(np.flatnonzero(B2[:,f]))):
                opposite.append(f)
            else:
                side.append(f)

        assert len(opposite)==1 and len(side)==4
        final=opposite[0]
        final_sign=-int(coeff[final])

        for perm in itertools.permutations(side):
            wf=np.asarray(perm,dtype=np.int16)
            ws=np.asarray([coeff[f] for f in perm],dtype=np.int8)
            out.append((c,final,final_sign,wf,ws))
    return out

cube_words=expected_cube_words(SEED_FACE)
gate("seed belongs to two cubes -> 48 temporal cube words",
     len(cube_words)==48,
     len(cube_words))

cube_amp_hist=Counter()
cube_state_hist=Counter()
cube_unsupported=[]
cube_history=Counter()

def perimeter_history(seed,wf,ws):
    q=B2[:,seed].astype(np.int16).copy()
    h=[]
    for f,s in zip(wf,ws):
        q+=int(s)*B2[:,int(f)].astype(np.int16)
        if len(h)<3:
            h.append(int(np.count_nonzero(q)))
    return tuple(h)

t0=time.time()
for c,ef,es,wf,ws in cube_words:
    factors=[face_steps(SEED_FACE,+1)]
    factors += [face_steps(int(f),int(s)) for f,s in zip(wf,ws)]
    factors += [face_steps(int(ef),-int(es))]  # bra endpoint

    amp,meta=contract_trace_network(factors,Nrank=N_RANK)
    if amp is None:
        cube_unsupported.append((c,ef,es,wf,ws,meta))
    else:
        cube_amp_hist[str(sp.factor(amp))]+=1
        cube_state_hist[meta["max_partition_states"]]+=1

    cube_history[perimeter_history(SEED_FACE,wf,ws)]+=1

print("cube explicit Haar amplitude histogram:",dict(cube_amp_hist))
print("cube DSU-state histogram:",dict(cube_state_hist))
print("cube perimeter-history histogram:",dict(cube_history))
print(f"contracted 48 six-trace cube networks in {time.time()-t0:.3f}s")

gate("all 48 cube trace networks explicitly contract",
     len(cube_unsupported)==0,
     len(cube_unsupported))
gate("all 48 cube surface networks contract exactly to N^-4",
     cube_amp_hist==Counter({"N**(-4)":48}),
     dict(cube_amp_hist))
gate("each cube network has a unique contraction state after local Haar integration",
     cube_state_hist==Counter({1:48}),
     dict(cube_state_hist))
gate("cube histories = 32x(6,6,6)+16x(6,8,6) across two cubes",
     cube_history==Counter({(6,6,6):32,(6,8,6):16}),
     dict(cube_history))

# Temporal resolvents.
def normalized_cube_weight(hist):
    w=Fraction(1,1)
    for perimeter in hist:
        w/=Fraction(4-perimeter,4)
    return w

bycube=defaultdict(list)
for c,ef,es,wf,ws in cube_words:
    bycube[c].append(normalized_cube_weight(perimeter_history(SEED_FACE,wf,ws)))

cube_sums={c:sum(vals,Fraction(0,1)) for c,vals in bycube.items()}
gate("24 temporal histories on each cube sum to -160",
     set(cube_sums.values())=={Fraction(-160,1)},
     cube_sums)

E0=(Nsym**2-1)/Nsym
cN=sp.factor(-160/(Nsym**4*E0**3))
alpha=sp.factor(-4*cN)

gate("c_N^square = -160/[N(N^2-1)^3]",
     sp.simplify(cN+160/(Nsym*(Nsym**2-1)**3))==0,
     cN)
gate("alpha_N = 640/[N(N^2-1)^3]",
     sp.simplify(alpha-640/(Nsym*(Nsym**2-1)**3))==0,
     alpha)
gate("SU(3) alpha_3 = 5/12",
     sp.simplify(alpha.subs(Nsym,3))==sp.Rational(5,12),
     sp.simplify(alpha.subs(Nsym,3)))

# ======================================================================================
# PART VII — DEPTH-3 SU(3) DETERMINANT FRONTIER
# ======================================================================================

print("\n"+"="*112)
print("PART VII — DEPTH-3 SU(3) EXPLICIT-CONTRACTION FRONTIER")
print("="*112)

wf3,ws3,flux3=generate_structured_words(3)
gate("depth-3 structured corpus has 53,160 linked words",
     len(wf3)==53160,
     len(wf3))

# Build per-word U/Ubar occupation counts from the actual plaquette sequence.
seed_vec=B2[:,SEED_FACE].astype(np.int16)

def word_occupations(wf,ws):
    q=seed_vec.copy()
    nF=(q>0).astype(np.uint8)
    nB=(q<0).astype(np.uint8)
    for f,s in zip(wf,ws):
        c=int(s)*B2[:,int(f)].astype(np.int16)
        q+=c
        nF+=(c>0).astype(np.uint8)
        nB+=(c<0).astype(np.uint8)
    return q,nF,nB

# Center-signature endpoint map.
endpoint_flux=[]
endpoint_meta=[]
for f in range(P):
    endpoint_flux.append(B2[:,f].astype(np.int16))
    endpoint_meta.append((f,+1))
    endpoint_flux.append(-B2[:,f].astype(np.int16))
    endpoint_meta.append((f,-1))

center_ep=defaultdict(list)
for ei,q in enumerate(endpoint_flux):
    center_ep[tuple(int(x%N_RANK) for x in q)].append(ei)

classification=Counter()
example_mixed=[]

for wf,ws,qword in zip(wf3,ws3,flux3):
    key=tuple(int(x%N_RANK) for x in qword)
    candidates=center_ep.get(key,())
    if not candidates:
        continue

    q,nF,nB=word_occupations(wf,ws)

    for ei in candidates:
        epq=endpoint_flux[ei]
        epF=(epq>0).astype(np.uint8)
        epB=(epq<0).astype(np.uint8)

        a=nF.astype(np.int16)+epB.astype(np.int16)
        b=nB.astype(np.int16)+epF.astype(np.int16)

        local_classes=[]
        for aa,bb in zip(a,b):
            if aa==0 and bb==0:
                continue
            local_classes.append(classify_local_occupancy(int(aa),int(bb),N_RANK))

        if "forbidden" in local_classes:
            classification["forbidden_bug"]+=1
        elif "mixed_determinant" in local_classes:
            classification["mixed_determinant_v05"]+=1
            if len(example_mixed)<3:
                example_mixed.append((endpoint_meta[ei],Counter(local_classes)))
        elif "balanced_high" in local_classes:
            classification["balanced_high_v05"]+=1
        elif "pure_epsilon" in local_classes:
            classification["v04_with_epsilon"]+=1
        else:
            classification["v04_balanced"]+=1

print("depth-3 center-compatible endpoint-pair classes:")
for k,v in classification.items():
    print(f"  {k:26s}: {v:,}")
if example_mixed:
    print("mixed determinant examples:",example_mixed)

gate("no center-compatible endpoint pair is locally representation-forbidden",
     classification["forbidden_bug"]==0,
     classification["forbidden_bug"])
gate("v0.4 explicitly identifies, rather than fakes, mixed determinant frontier",
     classification["mixed_determinant_v05"]>0,
     classification["mixed_determinant_v05"])

# ======================================================================================
# FINAL
# ======================================================================================

print("\n"+"="*112)
print("FINAL GATE SUMMARY")
print("="*112)

passed=sum(ok for _,ok,_ in gates)
for i,(name,ok,detail) in enumerate(gates,1):
    print(f"{i:02d}. {'PASS' if ok else 'FAIL'} — {name}" + (f" :: {detail}" if detail else ""))

print("-"*112)
print(f"PASSED {passed}/{len(gates)} GATES")

if passed==len(gates):
    print(r"""
RESULT — v0.4 EXPLICIT CONTRACTION ACCEPTANCE PASSED

This is no longer a boolean Haar sieve.

The engine explicitly materializes the local permutation intertwiners,
weights them with exact Weingarten rational functions, propagates the
Kronecker-delta identifications through the entire Wilson trace network,
and counts the remaining free color loops.

It derives, rather than inserts:

    plaquette norm                         = 1
    connected plaquette deformation       = 1/N
    six-face elementary-cube surface      = 1/N^4

The direct cube result therefore factorizes computationally as

    explicit Haar trace network  -> N^-4
    exact electric histories     -> -160/E0^3

giving

    c_N^square = -160/[N(N^2-1)^3].

The depth-2 corpus also demonstrates why a mere local-intertwiner count is
not enough: multiple local permutation channels recombine into exact global
trace-network amplitudes 1 or 2.

REMAINING v0.5 FRONTIER
-----------------------
Implement mixed SU(3) determinant-plus-delta intertwiners such as (4,1)
and their conjugates.  Those are now explicitly enumerated and quarantined;
no placeholder amplitude is used.
""")
else:
    print("\nAT LEAST ONE GATE FAILED. Do not advance to v0.5.")


<>:356: SyntaxWarning: invalid escape sequence '\d'
<>:356: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1124/2041197695.py:356: SyntaxWarning: invalid escape sequence '\d'
  (U^\dagger)_{ab} = \bar U_{ba}.


PART I — EXACT WEINGARTEN INTERTWINER TABLES
[PASS] k=1: exact Weingarten inverse satisfies G W = I
[PASS] k=2: exact Weingarten inverse satisfies G W = I
[PASS] k=3: exact Weingarten inverse satisfies G W = I
[PASS] k=2 identity Wg = 1/(N^2-1) :: 1/(N**2 - 1)
[PASS] k=2 transposition Wg = -1/[N(N^2-1)] :: -1/(N**3 - N)
k=2 Weingarten matrix:
⎡  1      -1   ⎤
⎢──────  ──────⎥
⎢ 2       3    ⎥
⎢N  - 1  N  - N⎥
⎢              ⎥
⎢ -1       1   ⎥
⎢──────  ──────⎥
⎢ 3       2    ⎥
⎣N  - N  N  - 1⎦
[PASS] SU(3) pure determinant primitive has six signed permutation branches :: [1/6, -1/6, -1/6, 1/6, 1/6, -1/6]
[PASS] SU(3) epsilon primitive gives integral det(U)=1

PART II — CUBIC CELL COMPLEX AND TRACE FACTORS
[PASS] B2 B3 = 0
[PASS] cubic face linked-neighborhood = 13 including self :: Counter({13: 81})

PART III — EXPLICIT TRACE-NETWORK CONTRACTION ENGINE

PART IV — FIRST-PRINCIPLES HAAR REGRESSIONS
plaquette norm = 1
[PASS] single fundamental plaquette Haar norm = 1 :: 1
deformation test: